In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, accuracy_score, f1_score
    
# Resolve project directories
PROJECT_ROOT = Path.cwd() if Path.cwd().name == "transit-delay-predictor" else Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "phase1_cleaned_transit_dataset.csv"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Define feature matrix and targets
features = [
    "hour_of_day", "day_of_week", "is_weekend", "is_peak_hour",
    "segment_id", "segment_distance_km", "scheduled_time_min",
    "historical_segment_avg_min", "rolling_3trip_delay_min",
    "prev_stop_dwell_sec", "weather_severity", "gps_speed_kmh"
]
target_reg = "delay_minutes"
target_clf = "delay_status"

# Time-respecting split: Days 1-24 Train (80%), Days 25-30 Test (20%)
train_mask = df["day_index"] <= 24
test_mask = df["day_index"] > 24

X_train, y_train_reg, y_train_clf = df.loc[train_mask, features], df.loc[train_mask, target_reg], df.loc[train_mask, target_clf]
X_test, y_test_reg, y_test_clf = df.loc[test_mask, features], df.loc[test_mask, target_reg], df.loc[test_mask, target_clf]

print(f"Training Window: Days 1 to 24 ({len(X_train)} samples)")
print(f"Testing Window:  Days 25 to 30 ({len(X_test)} samples)")

Training Window: Days 1 to 24 (3750 samples)
Testing Window:  Days 25 to 30 (945 samples)


In [6]:
# 1. Naive Baseline (Historical segment average delay)
naive_pred = X_test["historical_segment_avg_min"] - X_test["scheduled_time_min"]
naive_mae = mean_absolute_error(y_test_reg, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(y_test_reg, naive_pred))

# 2. 5-Fold TimeSeriesSplit Cross-Validation
tscv = TimeSeriesSplit(n_splits=5)
cv_maes = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    cv_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
    cv_model.fit(X_train.iloc[train_idx], y_train_reg.iloc[train_idx])
    pred = cv_model.predict(X_train.iloc[val_idx])
    fold_mae = mean_absolute_error(y_train_reg.iloc[val_idx], pred)
    cv_maes.append(fold_mae)
    print(f"Fold {fold} MAE: {fold_mae:.3f} mins")

print(f"\nMean TimeSeries CV MAE: {np.mean(cv_maes):.3f} mins")
print(f"Naive Baseline Test MAE: {naive_mae:.3f} mins (RMSE: {naive_rmse:.3f} mins)")

Fold 1 MAE: 0.788 mins
Fold 2 MAE: 0.677 mins
Fold 3 MAE: 0.807 mins
Fold 4 MAE: 0.644 mins
Fold 5 MAE: 0.640 mins

Mean TimeSeries CV MAE: 0.711 mins
Naive Baseline Test MAE: 3.498 mins (RMSE: 4.539 mins)


In [7]:
# Train primary regression model
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train_reg)

# Train quantile regressors for uncertainty intervals (Objective 3)
q_lower = GradientBoostingRegressor(loss="quantile", alpha=0.10, random_state=42)
q_upper = GradientBoostingRegressor(loss="quantile", alpha=0.90, random_state=42)
q_lower.fit(X_train, y_train_reg)
q_upper.fit(X_train, y_train_reg)

# Train supervised classification model
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train_clf)

# Evaluate on strictly held-out future test window (Days 25-30)
y_pred_reg = rf_reg.predict(X_test)
y_pred_lower = q_lower.predict(X_test)
y_pred_upper = q_upper.predict(X_test)
y_pred_clf = rf_clf.predict(X_test)

reg_mae = mean_absolute_error(y_test_reg, y_pred_reg)
reg_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
reg_r2 = r2_score(y_test_reg, y_pred_reg)
error_reduction = ((naive_mae - reg_mae) / naive_mae) * 100

print("=== REGRESSION RESULTS (TEST SET: DAYS 25-30) ===")
print(f"Random Forest MAE:  {reg_mae:.3f} mins")
print(f"Random Forest RMSE: {reg_rmse:.3f} mins")
print(f"R-squared (R2):     {reg_r2:.3f}")
print(f"Error Reduction vs Naive Baseline: {error_reduction:.2f}%\n")

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test_clf, y_pred_clf, target_names=["On-Time", "Moderate Delay", "Severe Delay"]))

=== REGRESSION RESULTS (TEST SET: DAYS 25-30) ===
Random Forest MAE:  0.627 mins
Random Forest RMSE: 0.914 mins
R-squared (R2):     0.945
Error Reduction vs Naive Baseline: 82.07%

=== CLASSIFICATION REPORT ===
                precision    recall  f1-score   support

       On-Time       0.93      0.89      0.91       280
Moderate Delay       0.86      0.85      0.86       358
  Severe Delay       0.90      0.93      0.92       307

      accuracy                           0.89       945
     macro avg       0.89      0.89      0.89       945
  weighted avg       0.89      0.89      0.89       945



In [8]:
# Measure peak vs off-peak prediction interval width
test_df = X_test.copy()
test_df["actual_delay"] = y_test_reg
test_df["predicted_delay"] = y_pred_reg
test_df["abs_error"] = np.abs(test_df["actual_delay"] - test_df["predicted_delay"])
test_df["uncertainty_spread"] = y_pred_upper - y_pred_lower

uncertainty_summary = test_df.groupby("is_peak_hour")[["abs_error", "uncertainty_spread"]].mean()
print("=== PEAK VS OFF-PEAK UNCERTAINTY (OBJ 3 BASIS) ===")
print(uncertainty_summary.round(3))

# Feature importances for XAI
importances = pd.Series(rf_reg.feature_importances_, index=features).sort_values(ascending=False)
print("\n=== TOP FEATURE IMPORTANCES (SHAP/XAI BASIS) ===")
print(importances.head(6).round(4))

# Save models to models/ directory
joblib.dump(rf_reg, MODELS_DIR / "delay_regressor.joblib")
joblib.dump(rf_clf, MODELS_DIR / "delay_classifier.joblib")
joblib.dump(q_lower, MODELS_DIR / "quantile_lower_10.joblib")
joblib.dump(q_upper, MODELS_DIR / "quantile_upper_90.joblib")

# Save evaluation metadata for frontend consumption
metadata = {
    "metrics": {
        "reg_mae": round(float(reg_mae), 3),
        "reg_rmse": round(float(reg_rmse), 3),
        "reg_r2": round(float(reg_r2), 3),
        "baseline_mae": round(float(naive_mae), 3),
        "error_reduction_pct": round(float(error_reduction), 2),
        "clf_accuracy": round(float(accuracy_score(y_test_clf, y_pred_clf)), 3),
        "clf_macro_f1": round(float(f1_score(y_test_clf, y_pred_clf, average='macro')), 3)
    },
    "uncertainty": {
        "off_peak_interval_width": round(float(uncertainty_summary.loc[0, "uncertainty_spread"]), 3),
        "peak_hour_interval_width": round(float(uncertainty_summary.loc[1, "uncertainty_spread"]), 3)
    },
    "feature_importances": importances.to_dict()
}

with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nTrained models and model_metadata.json successfully exported to models/")

=== PEAK VS OFF-PEAK UNCERTAINTY (OBJ 3 BASIS) ===
              abs_error  uncertainty_spread
is_peak_hour                               
0                 0.354               1.095
1                 1.131               3.491

=== TOP FEATURE IMPORTANCES (SHAP/XAI BASIS) ===
rolling_3trip_delay_min    0.6035
is_peak_hour               0.2335
weather_severity           0.0606
segment_id                 0.0388
gps_speed_kmh              0.0367
prev_stop_dwell_sec        0.0117
dtype: float64

Trained models and model_metadata.json successfully exported to models/
